# Penjualan antara antarprovinsi

ditulis: tim PSE Atha Bintang Wahyu M. | PSE UGM

--> IRIO, 34 provinsi × 17 sektor, peta sebelum pemekaran.
--> Penggunaan antara, triliun rupiah, transaksi RAS.
--> Objek gambar: penjualan yang dibeli provinsi lain.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

def find_root():
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "02 Code" / "BKPM" / "2026bkpm.Rproj").exists():
            return cand
    return Path("/Users/athamawardi/Desktop/Research-Projects/RDSI_Kemeninves")

ROOT = find_root()
sys.path.insert(0, str(ROOT / "02 Code" / "IO-IRIO-NA"))
import vizstyle as vs
from vizstyle import (
    NAVY, TEAL, RUST, GRAY, PAPER, CREAM, CMAP_FLOW, CMAP_LEAVE,
    apply_style, clean, head, save, ink_for, path_figure, SHORT, ISLAND_ORDER,
    ISLAND_COLOR, ISLAND_SHORT, TRILLION,
)

apply_style()
flows = vs.load_flows()
panel = vs.load_strength_panel()
absorb = vs.province_absorption(flows)
local_share = flows.loc[flows.same_province, "value"].sum() / flows.value.sum()
assert abs(local_share - 0.76398) < 1e-3
assert set(absorb.province) == set(panel.province)
print("local share", round(local_share, 3), "provinces", absorb.province.nunique())

## Koridor terbesar


### Gambar 1. Aliran antarprovinsi terbesar
--> Setiap batang adalah jumlah penjualan antara dari satu provinsi ke provinsi lain pada 2025, untuk seluruh 17 sektor.
--> Batang biru tua tetap di dalam Jawa.
--> Batang hijau toska punya paling sedikit satu ujung di luar Jawa.


In [ ]:
pair = (flows.loc[~flows.same_province]
        .groupby(["origin_province_name", "dest_province_name", "origin_island", "dest_island"], as_index=False)
        .value.sum())
top = pair.nlargest(15, "value").iloc[::-1]
top["java"] = top.origin_island.eq("Java") & top.dest_island.eq("Java")
fig, ax = plt.subplots(figsize=(10.2, 7.2))
y = np.arange(len(top))
ax.barh(y, top.value / TRILLION, color=np.where(top.java, NAVY, TEAL), height=0.72)
ax.set_yticks(y)
ax.set_yticklabels([f"{SHORT[a]}  to  {SHORT[b]}" for a, b in zip(top.origin_province_name, top.dest_province_name)],
                   fontsize=10)
ax.set_xlabel("Triliun rupiah")
clean(ax, grid="x")
handles = [
    Line2D([0], [0], color=NAVY, lw=6, label="Jawa ke Jawa"),
    Line2D([0], [0], color=TEAL, lw=6, label="Melibatkan pulau lain"),
]
ax.legend(handles=handles, frameon=False, fontsize=10, loc="lower right")
head(fig, "Lima belas aliran antara antarprovinsi terbesar, 2025",
     "Penggunaan antara, transaksi RAS, dalam triliun rupiah. Diurutkan dari nilai terbesar.")
save(fig, "02_top_corridors.png")

## Jalur dari 2017 ke 2025
--> Garis datar yang samar adalah tingkat 2017.
--> Garis karat adalah 2020.
--> Setiap gambar memakai dua garis yang sama dan skala vertikal yang sama.
--> Biru tua adalah yang dijual provinsi ke provinsi lain.


### Gambar 2. Provinsi acuan


In [ ]:
REF = [
    "DKI Jakarta", "Jawa Barat", "Jawa Tengah", "Jawa Timur",
    "Sumatera Utara", "Sumatera Selatan", "Kalimantan Timur", "Sulawesi Tengah",
]
path_figure(
    panel, REF, 4,
    "Indeks aliran antarprovinsi, Jawa, 2017–2025",
    "Indeks 2017 = 100. Sumbu 60 sampai 580, sama pada setiap gambar jalur. Garis penanda tahun 2020. Transaksi RAS, penggunaan antara.",
    "02_paths_reference.png",
)

### Gambar 3. Sulawesi


In [ ]:
SUL = [
    "Sulawesi Utara", "Sulawesi Tengah", "Sulawesi Selatan",
    "Sulawesi Tenggara", "Gorontalo", "Sulawesi Barat",
]
path_figure(
    panel, SUL, 3,
    "Indeks aliran antarprovinsi, Sulawesi, 2017–2025",
    "Indeks 2017 = 100. Sumbu 60 sampai 580, sama pada setiap gambar jalur. Garis penanda tahun 2020. Transaksi RAS, penggunaan antara.",
    "02_paths_sulawesi.png",
)

### Gambar 4. Maluku dan Papua


In [ ]:
MALPAP = ["Maluku", "Maluku Utara", "Papua", "Papua Barat"]
path_figure(
    panel, MALPAP, 4,
    "Indeks aliran antarprovinsi, Maluku dan Papua, 2017–2025",
    "Indeks 2017 = 100. Sumbu 60 sampai 580, sama pada setiap gambar jalur. Garis penanda tahun 2020. Transaksi RAS, penggunaan antara.",
    "02_paths_maluku_papua.png",
)

### Gambar 5. Bali dan Nusa Tenggara


In [ ]:
NUSA = ["Bali", "Nusa Tenggara Barat", "Nusa Tenggara Timur"]
path_figure(
    panel, NUSA, 3,
    "Indeks aliran antarprovinsi, Bali dan Nusa Tenggara, 2017–2025",
    "Indeks 2017 = 100. Sumbu 60 sampai 580, sama pada setiap gambar jalur. Garis penanda tahun 2020. Transaksi RAS, penggunaan antara.",
    "02_paths_nusa.png",
)

## Tujuan penjualan antara
--> Tiga pangsa yang sama digambar dua kali.
--> Gambar pulau adalah ringkasannya.
--> Gambar provinsi memakai warna yang sama dengan gambar pulau.
--> Provinsi yang sama berarti pembeli adalah penjual.
--> Pulau yang sama berarti provinsi lain di pulau itu.
--> Pulau lain berarti pembeli berada di tempat lain di kepulauan.


### Gambar 6. Pulau


In [ ]:
island = (absorb.groupby("island")[["local", "intra", "inter", "total"]].sum()
          .reindex(ISLAND_ORDER))
for col in ("local", "intra", "inter"):
    island[col + "_s"] = island[col] / island.total
parts = [
    ("local_s", NAVY, "Provinsi yang sama"),
    ("intra_s", TEAL, "Other province, same island"),
    ("inter_s", RUST, "Pulau lain"),
]
fig, ax = plt.subplots(figsize=(10.4, 5.4))
y = np.arange(len(island))[::-1]
left = np.zeros(len(island))
for col, color, label in parts:
    vals = island[col].to_numpy()
    ax.barh(y, vals, left=left, color=color, height=0.66, label=label)
    for i, (share, base) in enumerate(zip(vals, left)):
        if share >= 0.08:
            ink = "white" if color in (NAVY, RUST) else NAVY
            ax.text(base + share / 2, y[i], f"{share:.0%}", ha="center", va="center",
                    fontsize=8.5, color=ink)
    left = left + vals
ax.set_yticks(y)
ax.set_yticklabels([ISLAND_SHORT[i] for i in island.index], fontsize=11)
ax.set_xlim(0, 1)
ax.set_xlabel("Pangsa penjualan antara pulau")
clean(ax, grid="x")
ax.legend(frameon=False, fontsize=10, loc="lower right", bbox_to_anchor=(1.0, 1.02), ncol=3)
head(fig, "Komposisi penjualan antara menurut pulau, 2025",
     "Pangsa penjualan antara, transaksi RAS. Hijau adalah provinsi lain di pulau yang sama. Merah adalah pulau lain. Sisanya dibeli di provinsi penjual.",
     top=0.80)
save(fig, "02_island_absorption.png")
print(island[["local_s", "intra_s", "inter_s"]].round(3))

### Gambar 7. Provinsi


In [ ]:
plot = absorb.copy()
plot["island"] = pd.Categorical(plot.island, ISLAND_ORDER, ordered=True)
plot = plot.sort_values(["island", "inter_share"], ascending=[True, False])
parts = [
    ("local_share", NAVY, "Provinsi yang sama"),
    ("intra_share", TEAL, "Other province, same island"),
    ("inter_share", RUST, "Pulau lain"),
]
fig, ax = plt.subplots(figsize=(10.2, 11.0))
y = np.arange(len(plot))
left = np.zeros(len(plot))
for col, color, label in parts:
    vals = plot[col].to_numpy()
    ax.barh(y, vals, left=left, color=color, height=0.74, label=label)
    left = left + vals
ax.set_yticks(y)
ax.set_yticklabels([SHORT[p] for p in plot.province], fontsize=8.5)
for tick, island in zip(ax.get_yticklabels(), plot.island):
    tick.set_color(ISLAND_COLOR[island])
ax.set_xlim(0, 1)
ax.invert_yaxis()
ax.set_xlabel("Pangsa penjualan antara provinsi")
clean(ax, grid="x")
ax.legend(frameon=False, fontsize=10, ncol=3, loc="lower right", bbox_to_anchor=(1.0, 1.01))
head(fig, "Komposisi penjualan antara menurut provinsi, 2025",
     "Tiga pangsa yang sama dengan gambar pulau, transaksi RAS. Warna adalah pulau. Di dalam setiap pulau, provinsi diurutkan menurut pangsa penjualan ke pulau lain.",
     top=0.80)
save(fig, "02_province_absorption.png")

### Gambar 8. Matriks antarpulau
--> Diagonal dikosongkan dengan sengaja: penjualan yang tinggal di pulau sudah ada di kotak-kotak itu.


In [ ]:
off = flows.loc[flows.other_island]
mat = (off.pivot_table(index="origin_island", columns="dest_island", values="value", aggfunc="sum")
       .reindex(index=ISLAND_ORDER, columns=ISLAND_ORDER))
mat = mat / TRILLION
for name in ISLAND_ORDER:
    mat.loc[name, name] = np.nan
cmap = CMAP_FLOW.copy()
cmap.set_bad("#F7F4EF")
show = mat.to_numpy()
vmax = np.nanmax(show)
fig, ax = plt.subplots(figsize=(9.2, 7.4))
im = ax.imshow(np.ma.masked_invalid(show), cmap=cmap, vmin=0, vmax=vmax, aspect="equal")
for i in range(show.shape[0]):
    for j in range(show.shape[1]):
        val = show[i, j]
        text = "—" if np.isnan(val) else f"{val:.0f}"
        ax.text(j, i, text, ha="center", va="center", fontsize=9,
                color=GRAY if np.isnan(val) else ink_for(cmap, 0 if np.isnan(val) else val, 0, vmax))
ax.set_xticks(range(len(ISLAND_ORDER)))
ax.set_yticks(range(len(ISLAND_ORDER)))
ax.set_xticklabels([ISLAND_SHORT[i] for i in ISLAND_ORDER], rotation=40, ha="right")
ax.set_yticklabels([ISLAND_SHORT[i] for i in ISLAND_ORDER])
ax.tick_params(length=0)
ax.set_xlabel("Pulau pembeli")
ax.set_ylabel("Pulau penjual")
head(fig, "Matriks penjualan antara antarpulau, 2025",
     "Triliun rupiah, transaksi RAS. Baris menjual, kolom membeli. Diagonal kosong karena penjualan di dalam pulau tidak dimasukkan.")
save(fig, "02_island_destination_matrix.png")

## Pangsa dibeli di luar provinsi
--> Label pada tiap batang adalah pembeli luar yang terbesar dan nilai satu koridor itu.
--> Produk primer adalah pertanian dan pertambangan.
--> Industri pengolahan adalah pemrosesan.


### Gambar 9. Pangsa yang keluar, menurut provinsi


In [ ]:
import geopandas as gpd

g = gpd.read_file(vs.GADM)[["NAME_1", "geometry"]]
inv = {v: k for k, v in vs.GADM_NAME.items()}
g["province"] = g.NAME_1.map(inv)
assert g.province.notna().all() and set(g.province) == set(absorb.province)
geo = g.merge(absorb, on="province", how="left")
assert geo.primary_leave.notna().all()

fig, axes = plt.subplots(1, 2, figsize=(12.4, 5.6))
for ax, col, title in zip(
    axes,
    ["primary_leave", "manufacturing_leave"],
    ["Produk primer", "Industri pengolahan"],
):
    geo.plot(column=col, ax=ax, cmap=CMAP_LEAVE, vmin=0, vmax=0.85,
             edgecolor="#E2E5EA", linewidth=0.3, legend=False)
    ax.set_axis_off()
    ax.set_title(title, loc="left", fontsize=13)
    ax.set_xlim(94.5, 141.0)
    ax.set_ylim(-11, 6.2)
sm = plt.cm.ScalarMappable(cmap=CMAP_LEAVE, norm=plt.Normalize(0, 0.85))
sm.set_array([])
head(fig, "Pangsa penjualan yang dibeli di luar provinsi, 2025",
     "Pangsa penjualan antara, transaksi RAS. Kiri pertanian dan pertambangan, kanan industri pengolahan. Kedua peta memakai skala 0 sampai 0,85.",
     top=0.82)
# tight_layout has already run. Pull the maps left and hang the scale in the margin.
poses = [ax.get_position() for ax in axes]
span = poses[-1].x1 - poses[0].x0
scale = (span - 0.12) / span
origin = poses[0].x0
for ax, pos in zip(axes, poses):
    ax.set_position([
        origin + (pos.x0 - origin) * scale,
        pos.y0,
        pos.width * scale,
        pos.height,
    ])
bar_left = origin + (span - 0.12) + 0.02
cax = fig.add_axes([bar_left, poses[0].y0, 0.014, poses[0].height])
cbar = fig.colorbar(sm, cax=cax)
cbar.ax.tick_params(length=0, labelsize=8, colors=GRAY)
cbar.set_label("Pangsa yang dibeli provinsi lain", color=GRAY, fontsize=9)
cbar.outline.set_visible(False)
save(fig, "02_leave_share_maps.png")


### Gambar 10. Produk primer, dan provinsi yang membelinya


In [ ]:
buy_p = vs.top_buyer(flows, "Primary")
prim = absorb.merge(buy_p, on="province")
plot = prim.sort_values("primary_leave", ascending=True)
fig, ax = plt.subplots(figsize=(10.6, 9.8))
y = np.arange(len(plot))
ax.barh(y, plot.primary_leave, color=plot.island.map(ISLAND_COLOR), height=0.72)
for i, r in enumerate(plot.itertuples()):
    tr = r.buyer_value / TRILLION
    value = f"{tr:.0f} trn" if tr >= 10 else f"{tr:.1f} trn"
    ax.text(r.primary_leave + 0.015, i,
            f"{SHORT[r.buyer]}   {value}",
            va="center", ha="left", fontsize=8, color=GRAY)
ax.set_yticks(y)
ax.set_yticklabels([SHORT[p] for p in plot.province], fontsize=8.5)
ax.set_xlim(0, 1.15)
ax.set_xlabel("Pangsa penjualan primer yang dibeli provinsi lain")
clean(ax, grid="x")
head(fig, "Pangsa penjualan primer yang dibeli provinsi lain, 2025",
     "Pertanian dan pertambangan, transaksi RAS. Panjang batang adalah pangsa yang keluar. Label adalah pembeli terbesar dan nilai koridor itu, dalam triliun rupiah.")
save(fig, "02_primary_buyers.png")
print(plot.nlargest(6, "primary_leave")[["province", "primary_leave", "buyer", "buyer_value"]].assign(
    tr=lambda d: (d.buyer_value / TRILLION).round(1)).round(3).to_string(index=False))

### Gambar 11. Industri pengolahan, dan provinsi yang membelinya


In [ ]:
buy_m = vs.top_buyer(flows, "Manufacturing")
man = absorb.merge(buy_m, on="province")
plot = man.sort_values("manufacturing_leave", ascending=True)
fig, ax = plt.subplots(figsize=(10.6, 9.8))
y = np.arange(len(plot))
ax.barh(y, plot.manufacturing_leave, color=plot.island.map(ISLAND_COLOR), height=0.72)
for i, r in enumerate(plot.itertuples()):
    tr = r.buyer_value / TRILLION
    value = f"{tr:.0f} trn" if tr >= 10 else f"{tr:.1f} trn"
    ax.text(r.manufacturing_leave + 0.015, i,
            f"{SHORT[r.buyer]}   {value}",
            va="center", ha="left", fontsize=8, color=GRAY)
ax.set_yticks(y)
ax.set_yticklabels([SHORT[p] for p in plot.province], fontsize=8.5)
ax.set_xlim(0, 1.22)
ax.set_xlabel("Pangsa penjualan industri yang dibeli provinsi lain")
clean(ax, grid="x")
head(fig, "Pangsa penjualan industri pengolahan yang dibeli provinsi lain, 2025",
     "Transaksi RAS. Panjang batang adalah pangsa yang keluar. Label adalah pembeli terbesar dan nilai koridor itu, dalam triliun rupiah.")
save(fig, "02_manufacturing_buyers.png")

### Aliran primer, sekunder, dan tersier, lalu pembelian tiap komoditas
--> Peta aliran ada tiga.
--> Primer adalah pertanian dan pertambangan.
--> Sekunder adalah industri pengolahan, listrik, air, dan konstruksi.
--> Tersier adalah perdagangan dan jasa.
--> Pembeli yang digambar adalah pembeli utama tiap komoditas: dua pembeli terbesar, pembeli yang mengambil sekurang-kurangnya seperlima penjualan ke luar provinsi, dan, di luar Jawa, koridor yang paling sedikit Rp 5 triliun.
--> Setiap panel adalah satu pulau penjual.


In [ ]:
kept = vs.group_flow_maps(flows)
vs.commodity_purchase_bars(flows)

nm = flows.loc[
    flows.origin_province_name.eq("Maluku Utara") & ~flows.same_province & flows.value.gt(0)
]
nm = (nm.groupby(["origin_sector_code", "dest_province_name", "dest_island"], as_index=False)
      .value.sum())
nm = nm.loc[nm.value.ge(0.05 * TRILLION)].sort_values(
    ["origin_sector_code", "value"], ascending=[True, False]
)
nm["commodity"] = nm.origin_sector_code.map(vs.SECTOR_LABEL)
nm["ke"] = nm.dest_province_name.map(SHORT)
nm["pulau"] = nm.dest_island.map(vs.ISLAND_SHORT)
nm["trn"] = (nm.value / TRILLION).round(2)
print("Maluku Utara, koridor paling sedikit Rp 0,05 triliun")
print(nm[["commodity", "ke", "pulau", "trn"]].to_string(index=False))


## Tujuan penjualan menurut komoditas


### Gambar 14. Tujuan pembelian menurut komoditas


In [ ]:
sec = vs.load_sector_spillover()
sec = sec.loc[sec.year.eq(2025)].sort_values("away", ascending=True)
fig, axes = plt.subplots(1, 2, figsize=(12.2, 7.6), sharey=True)
y = np.arange(len(sec))
labels = [vs.SECTOR_LABEL[c] for c in sec.origin_sector_code]

axes[0].barh(y, sec.local / sec.total, color=NAVY, height=0.72, label="Provinsi yang sama")
axes[0].barh(y, sec.intra / sec.total, left=sec.local / sec.total, color=TEAL, height=0.72, label="Pulau yang sama, provinsi lain")
axes[0].barh(y, sec.inter / sec.total, left=(sec.local + sec.intra) / sec.total, color=RUST, height=0.72, label="Pulau lain")
axes[0].set_xlim(0, 1)
axes[0].set_xlabel("Pangsa dari penjualan komoditas itu sendiri")
clean(axes[0], grid="x")

axes[1].barh(y, sec.intra / vs.TRILLION, color=TEAL, height=0.72, label="Pulau yang sama")
axes[1].barh(y, sec.inter / vs.TRILLION, left=sec.intra / vs.TRILLION, color=RUST, height=0.72, label="Pulau lain")
axes[1].set_xlabel("Triliun rupiah yang dijual ke provinsi lain")
clean(axes[1], grid="x")
handles = [
    Line2D([0], [0], color=NAVY, lw=6, label="Provinsi yang sama"),
    Line2D([0], [0], color=TEAL, lw=6, label="Pulau yang sama, provinsi lain"),
    Line2D([0], [0], color=RUST, lw=6, label="Pulau lain"),
]
fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.0),
           frameon=False, fontsize=9, ncol=3)

axes[0].set_yticks(y)
axes[0].set_yticklabels(labels, fontsize=9)
head(fig, "Arah penjualan antara menurut komoditas, 2025",
     "Transaksi RAS. Kiri: pangsa penjualan komoditas itu, berjumlah satu. Kanan: nilai yang dijual ke luar provinsi, dalam triliun rupiah. Baris diurutkan menurut nilai ke luar provinsi.",
     top=0.86)
save(fig, "02_sector_spillover.png")

## Peta Indonesia, seluruh penjualan komoditas

Peta pulau penjual di atas tetap, untuk dibaca kemudian.
Bagian ini satu peta Indonesia, letak provinsi yang sebenarnya.
Tiap peta menggambar seluruh penjualan ke provinsi lain: kelompok primer, sekunder, dan tersier, lalu tiap kode komoditas.
Tidak dipotong pada lima koridor terbesar. Lebar panah berskala di dalam peta itu sendiri. Warna adalah pulau pembeli.

In [ ]:
import geopandas as gpd

gadm = gpd.read_file(vs.GADM)[["NAME_1", "geometry"]]
gadm["province"] = gadm.NAME_1.map({v: k for k, v in vs.GADM_NAME.items()})
assert gadm.province.notna().all()
_pts = gadm.geometry.representative_point()
cent = {prov: (float(x), float(y)) for prov, x, y in zip(gadm.province, _pts.x, _pts.y)}
away = flows.loc[~flows.same_province & flows.value.gt(0)].copy()

def koridor(frame):
    return frame.groupby(
        ["origin_province_name", "dest_province_name", "origin_island", "dest_island"],
        as_index=False,
    ).value.sum()

WARNA_KOMODITAS = {
    "A": "#1F7A4D", "B": "#C47B2B", "C": "#0E2440", "D": "#C4A35A", "E": "#5BA3C4",
    "F": "#A35A3A", "G": "#46A694", "H": "#3D6B8C", "I": "#D9042B", "J": "#1A6F8A",
    "K": "#797F8C", "L": "#A33B73", "MN": "#6B4C7A", "O": "#2F6F4F", "P": "#8C3A3A",
    "Q": "#E07A5F", "RSTU": "#4C6B8A",
}
NAMA_PENDEK = {
    "A": "A Pertanian", "B": "B Pertambangan", "C": "C Industri", "D": "D Listrik",
    "E": "E Air", "F": "F Konstruksi", "G": "G Perdagangan", "H": "H Transportasi",
    "I": "I Akomodasi", "J": "J Informasi", "K": "K Keuangan", "L": "L Real estat",
    "MN": "MN Jasa perusahaan", "O": "O Administrasi", "P": "P Pendidikan",
    "Q": "Q Kesehatan", "RSTU": "RSTU Jasa lain",
}

JAVA = {
    "DKI Jakarta": (-0.35, 1.05),
    "Banten": (-1.9, 0.4),
    "Jawa Barat": (-0.55, -1.2),
    "Jawa Tengah": (0.15, 0.95),
    "DI Yogyakarta": (0.35, -1.15),
    "Jawa Timur": (1.4, -0.7),
}

def letak_label(names):
    items = []
    for name in names:
        x, y = cent[name]
        if name in JAVA:
            dx, dy = JAVA[name]
        else:
            dx, dy = vs._NUDGE_DEG.get(name, (0.7, 0.35))
            dx, dy = dx * 0.75, dy * 0.75
        items.append([name, x, y, x + dx, y + dy])
    vs._spread_labels(items, 0.22)
    return items

def gambar_semua(pair, judul, anak, fname):
    fig, ax = plt.subplots(figsize=(12.8, 7.4))
    gadm.plot(ax=ax, color="#F7F4EF", edgecolor="#E2E5EA", linewidth=0.35)
    ax.set_axis_off()
    ax.set_xlim(93.2, 142.6)
    ax.set_ylim(-11.9, 7.3)
    ax.set_aspect("equal", adjustable="box")
    vmax = float(pair.value.max())
    for r in pair.sort_values("value").itertuples(index=False):
        a, b = cent[r.origin_province_name], cent[r.dest_province_name]
        dx, dy = b[0] - a[0], b[1] - a[1]
        span = (dx * dx + dy * dy) ** 0.5 or 1.0
        sign = 1 if r.origin_province_name < r.dest_province_name else -1
        rad = (0.16 if span < 6 else 0.07 if span < 15 else 0.03) * sign
        t = r.value / vmax
        ax.annotate(
            "", xy=b, xytext=a,
            arrowprops=dict(
                arrowstyle="-|>", color=vs.ISLAND_COLOR[r.dest_island],
                lw=0.12 + 2.8 * (t ** 0.55), alpha=0.05 + 0.9 * (t ** 0.4),
                mutation_scale=5 + 4 * t,
                connectionstyle=f"arc3,rad={rad}",
                shrinkA=2.5, shrinkB=2.5,
            ),
            zorder=2 + t,
        )
    for name, x, y, lx, ly in letak_label(sorted(gadm.province)):
        ax.scatter([x], [y], s=11, color=vs.ISLAND_COLOR[vs.ISLAND[name]], zorder=4, linewidths=0)
        ax.annotate(
            vs.MAP_LABEL[name], (lx, ly), fontsize=6.5, color=NAVY, ha="center", va="center",
            zorder=5, bbox=dict(facecolor="white", edgecolor="none", alpha=0.82, pad=0.08),
        )
    handles = [
        Line2D([0], [0], color=vs.ISLAND_COLOR[i], lw=2.2, label=vs.ISLAND_SHORT[i])
        for i in vs.ISLAND_ORDER
    ]
    fig.legend(handles=handles, frameon=False, fontsize=8, ncol=7, loc="lower center", bbox_to_anchor=(0.5, 0.0))
    head(fig, judul, anak, top=0.86)
    save(fig, fname)
    plt.close(fig)

for slug, title, codes, blurb in vs.FLOW_GROUPS:
    part = away.loc[away.origin_sector_code.isin(codes)]
    pair = koridor(part)
    gambar_semua(
        pair,
        f"Penjualan {title.lower()} ke provinsi lain, 2025",
        f"{blurb} Penggunaan antara, transaksi RAS. "
        "Setiap provinsi yang menjual ikut, tidak dipotong pada lima koridor terbesar. "
        f"{len(pair)} koridor, {pair.origin_province_name.nunique()} provinsi penjual, "
        f"{pair.value.sum() / TRILLION:.0f} triliun rupiah. "
        "Lebar dan kerapatan warna berskala di dalam peta ini. Warna adalah pulau pembeli.",
        f"02_nasional_{slug}.png",
    )
    print(f"{title}: {len(pair)} koridor, {pair.origin_province_name.nunique()} penjual")

for code in vs.SECTOR_ORDER:
    part = away.loc[away.origin_sector_code.eq(code)]
    if part.empty:
        continue
    pair = koridor(part)
    nama = vs.SECTOR_LABEL[code]
    gambar_semua(
        pair,
        f"Penjualan {nama.lower()} ke provinsi lain, 2025",
        f"Kode {code}. Penggunaan antara, transaksi RAS. "
        "Setiap provinsi yang menjual komoditas ini ikut, tidak dipotong pada lima koridor terbesar. "
        f"{len(pair)} koridor, {pair.origin_province_name.nunique()} provinsi penjual, "
        f"{pair.value.sum() / TRILLION:.0f} triliun rupiah. "
        "Lebar panah berskala di dalam peta ini, tidak dibandingkan dengan komoditas lain. "
        "Warna adalah pulau pembeli.",
        f"02_nasional_{code}.png",
    )
    print(f"{code} {nama}: {len(pair)} koridor, {pair.origin_province_name.nunique()} penjual")


## Tiap provinsi, lima dan sebelas koridor komoditas

Satu koridor adalah satu komoditas yang dijual satu provinsi ke provinsi lain.
Tiap provinsi menjaga koridor komoditasnya yang terbesar: lima pada peta pertama, sebelas pada peta kedua.
Warna garis adalah komoditas pada koridor itu, dan legenda hanya memuat komoditas yang tampil.
Peta pulau penjual dan peta seluruh penjualan di atas tidak diganti.

In [ ]:
komoditas = away.groupby(
    ["origin_province_name", "origin_sector_code", "dest_province_name", "origin_island", "dest_island"],
    as_index=False,
).value.sum()

def pilih_provinsi(n):
    urut = komoditas.sort_values("value", ascending=False)
    return urut.groupby("origin_province_name", group_keys=False).head(n)

def gambar_provinsi(kept, judul, anak, fname):
    gambar = kept.sort_values("value").copy()
    kunci = ["origin_province_name", "dest_province_name"]
    gambar["slot"] = gambar.groupby(kunci).cumcount()
    gambar["nslot"] = gambar.groupby(kunci)["origin_sector_code"].transform("size")
    fig, ax = plt.subplots(figsize=(12.8, 7.4))
    gadm.plot(ax=ax, color="#F7F4EF", edgecolor="#E2E5EA", linewidth=0.35)
    ax.set_axis_off()
    ax.set_xlim(93.2, 142.6)
    ax.set_ylim(-11.9, 7.3)
    ax.set_aspect("equal", adjustable="box")
    vmax = float(gambar.value.max())
    for r in gambar.itertuples(index=False):
        a, b = cent[r.origin_province_name], cent[r.dest_province_name]
        dx, dy = b[0] - a[0], b[1] - a[1]
        span = (dx * dx + dy * dy) ** 0.5 or 1.0
        sign = 1 if r.origin_province_name < r.dest_province_name else -1
        rad = (0.16 if span < 6 else 0.07 if span < 15 else 0.03) * sign
        rad += (r.slot - (r.nslot - 1) / 2) * 0.045
        t = r.value / vmax
        lw = 0.25 + 2.3 * (t ** 0.7)
        ax.annotate(
            "", xy=b, xytext=a,
            arrowprops=dict(
                arrowstyle="-|>", color=WARNA_KOMODITAS[r.origin_sector_code],
                lw=lw, alpha=0.28 + 0.7 * t,
                mutation_scale=6 + lw,
                connectionstyle=f"arc3,rad={rad}",
                shrinkA=3, shrinkB=3,
            ),
            zorder=2 + t,
        )
    for name, x, y, lx, ly in letak_label(sorted(gadm.province)):
        ax.scatter([x], [y], s=11, color=NAVY, zorder=4, linewidths=0)
        ax.annotate(
            vs.MAP_LABEL[name], (lx, ly), fontsize=6.5, color=NAVY, ha="center", va="center",
            zorder=5, bbox=dict(facecolor="white", edgecolor="none", alpha=0.82, pad=0.08),
        )
    hadir = [c for c in vs.SECTOR_ORDER if c in set(gambar.origin_sector_code)]
    handles = [
        Line2D([0], [0], color=WARNA_KOMODITAS[c], lw=2.2, label=NAMA_PENDEK[c])
        for c in hadir
    ]
    fig.legend(
        handles=handles, frameon=False, fontsize=8, ncol=5,
        loc="lower center", bbox_to_anchor=(0.5, 0.0),
    )
    head(fig, judul, anak, top=0.86)
    save(fig, fname)
    plt.close(fig)
    return gambar

for n, slug, kata in ((5, "5", "Lima"), (11, "11", "Sebelas")):
    kept = pilih_provinsi(n)
    gambar = gambar_provinsi(
        kept,
        f"{kata} koridor komoditas terbesar tiap provinsi, 2025",
        "Penggunaan antara, transaksi RAS. Tiap provinsi menjaga koridor komoditasnya yang terbesar: "
        "satu komoditas menuju satu provinsi lain. "
        f"{len(kept)} koridor komoditas, {kept.value.sum() / away.value.sum():.0%} dari penjualan ke luar provinsi. "
        "Lebar dan kerapatan warna berskala di dalam peta ini. Warna adalah komoditas pada garis itu.",
        f"02_nasional_provinsi_{slug}.png",
    )
    print(f"{kata}: {len(kept)} koridor komoditas")
    atas = kept.sort_values("value", ascending=False).head(12)
    print(pd.DataFrame({
        "dari": atas.origin_province_name.map(vs.MAP_LABEL),
        "komoditas": atas.origin_sector_code.map(NAMA_PENDEK),
        "ke": atas.dest_province_name.map(vs.MAP_LABEL),
        "trn": (atas.value / TRILLION).round(2),
    }).to_string(index=False))
    print()

## Lima komoditas terbesar, bertahap

Ini peta lain. Peta tiap provinsi di atas tetap.
Lima komoditas diurutkan dari penjualan antarprovinsi yang paling besar.
Tiap peta memberi tiap komoditas itu jumlah koridor yang sama, makin banyak pada peta berikutnya: lima, sebelas, dua puluh, empat puluh, lalu delapan puluh.
Warna adalah komoditas, sama dengan legenda.

In [ ]:
peringkat_komoditas = (
    komoditas.groupby("origin_sector_code").value.sum().sort_values(ascending=False)
)
lima_kode = list(peringkat_komoditas.head(5).index)
print("Lima komoditas terbesar:", ", ".join(NAMA_PENDEK[c] for c in lima_kode))

def gambar_kuota(n, judul, anak, fname):
    bagian = []
    for code in lima_kode:
        bagian.append(
            komoditas.loc[komoditas.origin_sector_code.eq(code)]
            .sort_values("value", ascending=False)
            .head(n)
        )
    gambar = pd.concat(bagian, ignore_index=True).sort_values("value")
    kunci = ["origin_province_name", "dest_province_name"]
    gambar["slot"] = gambar.groupby(kunci).cumcount()
    gambar["nslot"] = gambar.groupby(kunci)["origin_sector_code"].transform("size")
    fig, ax = plt.subplots(figsize=(12.8, 7.4))
    gadm.plot(ax=ax, color="#F7F4EF", edgecolor="#E2E5EA", linewidth=0.35)
    ax.set_axis_off()
    ax.set_xlim(93.2, 142.6)
    ax.set_ylim(-11.9, 7.3)
    ax.set_aspect("equal", adjustable="box")
    vmax = float(gambar.value.max())
    padat = min(1.0, (n - 5) / 75)
    for r in gambar.itertuples(index=False):
        a, b = cent[r.origin_province_name], cent[r.dest_province_name]
        dx, dy = b[0] - a[0], b[1] - a[1]
        span = (dx * dx + dy * dy) ** 0.5 or 1.0
        sign = 1 if r.origin_province_name < r.dest_province_name else -1
        rad = (0.22 if span < 6 else 0.09 if span < 15 else 0.04) * sign
        rad += (r.slot - (r.nslot - 1) / 2) * (0.05 - 0.03 * padat)
        t = r.value / vmax
        lw = (0.45 - 0.3 * padat) + 3.2 * (t ** (0.75 - 0.2 * padat))
        ax.annotate(
            "", xy=b, xytext=a,
            arrowprops=dict(
                arrowstyle="-|>", color=WARNA_KOMODITAS[r.origin_sector_code],
                lw=lw, alpha=(0.45 - 0.33 * padat) + (0.5 + 0.3 * padat) * t,
                mutation_scale=(7 - 2 * padat) + lw,
                connectionstyle=f"arc3,rad={rad}",
                shrinkA=3, shrinkB=3,
            ),
            zorder=2 + t,
        )
    nama = sorted(set(gambar.origin_province_name) | set(gambar.dest_province_name))
    for name, x, y, lx, ly in letak_label(nama):
        ax.scatter([x], [y], s=14, color=NAVY, zorder=4, linewidths=0)
        ax.annotate(
            vs.MAP_LABEL[name], (lx, ly), fontsize=7.5, color=NAVY, ha="center", va="center",
            zorder=5, bbox=dict(facecolor="white", edgecolor="none", alpha=0.9, pad=0.1),
        )
    handles = [
        Line2D([0], [0], color=WARNA_KOMODITAS[c], lw=2.4, label=NAMA_PENDEK[c])
        for c in lima_kode
    ]
    fig.legend(handles=handles, frameon=False, fontsize=8, ncol=5, loc="lower center", bbox_to_anchor=(0.5, 0.0))
    head(fig, judul, anak, top=0.86)
    save(fig, fname)
    plt.close(fig)
    return gambar.sort_values("value", ascending=False)

langkah = (
    (5, "5", "lima"),
    (11, "11", "sebelas"),
    (20, "20", "dua puluh"),
    (40, "40", "empat puluh"),
    (80, "80", "delapan puluh"),
)
for n, slug, kata in langkah:
    judul = f"Lima komoditas terbesar, {kata} koridor tiap komoditas, 2025"
    anak = (
        "Penggunaan antara, transaksi RAS. Lima komoditas diurutkan menurut penjualan ke provinsi lain: "
        + ", ".join(NAMA_PENDEK[c] for c in lima_kode)
        + f". Tiap komoditas menyumbang {kata} koridor terbesarnya, "
        f"{n * len(lima_kode)} garis. "
        "Lebar berskala di dalam peta ini. Warna adalah komoditas."
    )
    gambar = gambar_kuota(n, judul, anak, f"02_nasional_lima_komoditas_{slug}.png")
    pangsa = []
    for code in lima_kode:
        bagian = gambar.loc[gambar.origin_sector_code.eq(code), "value"].sum()
        pangsa.append(f"{NAMA_PENDEK[code]} {bagian / peringkat_komoditas[code]:.0%}")
    print(f"{kata}: " + ", ".join(pangsa))
    cetak = gambar if n <= 20 else gambar.head(12)
    print(pd.DataFrame({
        "dari": cetak.origin_province_name.map(vs.MAP_LABEL),
        "komoditas": cetak.origin_sector_code.map(NAMA_PENDEK),
        "ke": cetak.dest_province_name.map(vs.MAP_LABEL),
        "trn": (cetak.value / TRILLION).round(2),
    }).to_string(index=False))
    print()
